In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Load dataset
df = pd.read_csv("../data/processed/final_phishing_dataset.csv")

# Keep only needed columns
df = df[["url", "label"]].dropna()

# Clean
df["url"] = df["url"].astype(str).str.strip().str.lower()
df = df[df["url"] != ""]

print(df.shape)
print(df["label"].value_counts())

X = df["url"].values
y = df["label"].values

# Train 80%, validation 10%, test 10%
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print("Train:", len(X_train))
print("Val:", len(X_val))
print("Test:", len(X_test))

(99995, 2)
label
0    50000
1    49995
Name: count, dtype: int64
Train: 79996
Val: 9999
Test: 10000


In [3]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import pickle
import os

MAX_LEN = 200
EMBED_DIM = 64

tokenizer = Tokenizer(char_level=True, lower=True, oov_token="[UNK]")
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_val_seq = tokenizer.texts_to_sequences(X_val)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding="post", truncating="post")
X_val_pad = pad_sequences(X_val_seq, maxlen=MAX_LEN, padding="post", truncating="post")
X_test_pad = pad_sequences(X_test_seq, maxlen=MAX_LEN, padding="post", truncating="post")

vocab_size = len(tokenizer.word_index) + 1

print("Vocab size:", vocab_size)
print("Train shape:", X_train_pad.shape)

os.makedirs("models", exist_ok=True)
with open("models/url_tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

Vocab size: 69
Train shape: (79996, 200)


In [4]:
import tensorflow as tf
from tensorflow.keras.layers import (
    Input, Embedding, Conv1D, MaxPooling1D,
    Bidirectional, GRU, Dense, Dropout,
    Layer
)
from tensorflow.keras.models import Model

class AttentionLayer(Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def build(self, input_shape):
        self.W = self.add_weight(
            name="att_weight",
            shape=(input_shape[-1], 1),
            initializer="glorot_uniform",
            trainable=True
        )
        self.b = self.add_weight(
            name="att_bias",
            shape=(input_shape[1], 1),
            initializer="zeros",
            trainable=True
        )
        super().build(input_shape)

    def call(self, x):
        # x shape: (batch, timesteps, features)
        e = tf.keras.backend.tanh(tf.keras.backend.dot(x, self.W) + self.b)
        a = tf.keras.backend.softmax(e, axis=1)
        output = x * a
        return tf.keras.backend.sum(output, axis=1)

inputs = Input(shape=(MAX_LEN,))

x = Embedding(input_dim=vocab_size, output_dim=EMBED_DIM)(inputs)
x = Conv1D(filters=128, kernel_size=5, activation="relu")(x)
x = MaxPooling1D(pool_size=2)(x)
x = Bidirectional(GRU(64, return_sequences=True))(x)
x = AttentionLayer()(x)
x = Dense(64, activation="relu")(x)
x = Dropout(0.3)(x)
outputs = Dense(1, activation="sigmoid")(x)

model = Model(inputs=inputs, outputs=outputs)

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.Precision(), tf.keras.metrics.Recall()]
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 200)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 200, 64)        │         4,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 196, 128)       │        41,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 98, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 98, 128)        │        74,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ attention_layer                 │ (None, 128)            │           226 │
│ (AttentionLayer)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 128,547 (502.14 KB)

 Trainable params: 128,547 (502.14 KB)

 Non-trainable params: 0 (0.00 B)

In [5]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

callbacks = [
    EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2),
    ModelCheckpoint("models/best_phishing_model.keras", monitor="val_loss", save_best_only=True)
]

history = model.fit(
    X_train_pad, y_train,
    validation_data=(X_val_pad, y_val),
    epochs=10,
    batch_size=128,
    callbacks=callbacks
)

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 149s 223ms/step - accuracy: 0.9230 - loss: 0.1854 - precision: 0.9344 - recall: 0.9099 - val_accuracy: 0.9685 - val_loss: 0.0986 - val_precision: 0.9958 - val_recall: 0.9410 - learning_rate: 0.0010
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 131s 210ms/step - accuracy: 0.9785 - loss: 0.0649 - precision: 0.9822 - recall: 0.9746 - val_accuracy: 0.9827 - val_loss: 0.0481 - val_precision: 0.9918 - val_recall: 0.9734 - learning_rate: 0.0010
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 131s 210ms/step - accuracy: 0.9849 - loss: 0.0478 - precision: 0.9887 - recall: 0.9811 - val_accuracy: 0.9853 - val_loss: 0.0427 - val_precision: 0.9821 - val_recall: 0.9886 - learning_rate: 0.0010
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 124s 198ms/step - accuracy: 0.9878 - loss: 0.0388 - precision: 0.9907 - recall: 0.9847 - val_accuracy: 0.9877 - val_loss: 0.0373 - val_precision: 0.9923 - val_recall: 0.9830 - learning_rate: 0.0010
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 131s 210

In [6]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred_prob = model.predict(X_test_pad)
y_pred = (y_pred_prob >= 0.5).astype(int).flatten()

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

313/313 ━━━━━━━━━━━━━━━━━━━━ 12s 37ms/step
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      5000
           1       0.99      0.99      0.99      5000

    accuracy                           0.99     10000
   macro avg       0.99      0.99      0.99     10000
weighted avg       0.99      0.99      0.99     10000

[[4965   35]
 [  68 4932]]


In [7]:
model.save("models/final_phishing_model.keras")
print("Model saved.")

Model saved.


In [8]:
def predict_url(url):
    url = str(url).strip().lower()
    seq = tokenizer.texts_to_sequences([url])
    pad = pad_sequences(seq, maxlen=MAX_LEN, padding="post", truncating="post")
    
    prob = model.predict(pad, verbose=0)[0][0]
    label = 1 if prob >= 0.5 else 0
    
    if label == 1:
        print(f"Phishing probability: {prob:.4f} -> PHISHING")
    else:
        print(f"Phishing probability: {prob:.4f} -> LEGITIMATE")

# Test examples
predict_url("https://google.com")
predict_url("https://stream-billing.com/login")

Phishing probability: 0.0021 -> LEGITIMATE
Phishing probability: 1.0000 -> PHISHING


In [9]:
predict_url("https://www.google.com")
predict_url("https://www.facebook.com")
predict_url("https://www.amazon.co.uk")
predict_url("https://www.bbc.com/news")
predict_url("https://github.com")
predict_url("https://stackoverflow.com/questions")
predict_url("https://www.apple.com/uk")
predict_url("https://www.microsoft.com/en-gb")
predict_url("https://www.nhs.uk")
predict_url("https://www.gov.uk")

Phishing probability: 0.0015 -> LEGITIMATE
Phishing probability: 0.0004 -> LEGITIMATE
Phishing probability: 0.0001 -> LEGITIMATE
Phishing probability: 0.0001 -> LEGITIMATE
Phishing probability: 0.0022 -> LEGITIMATE
Phishing probability: 0.0828 -> LEGITIMATE
Phishing probability: 0.0004 -> LEGITIMATE
Phishing probability: 0.0002 -> LEGITIMATE
Phishing probability: 0.0001 -> LEGITIMATE
Phishing probability: 0.0001 -> LEGITIMATE


In [10]:
predict_url("http://secure-login-paypal.com")
predict_url("http://paypal.verify-user-account.com")
predict_url("http://bankofamerica.secure-update-login.com")
predict_url("http://amazon-login.security-check.net")
predict_url("http://verify-your-account-now.com/login")
predict_url("http://account-update-alert-paypal.com")
predict_url("http://apple-id-confirmation-secure.com")
predict_url("http://login-facebook-security-alert.com")
predict_url("http://secure-update-banking-info.com")
predict_url("http://your-account-has-been-suspended.com")

Phishing probability: 1.0000 -> PHISHING
Phishing probability: 1.0000 -> PHISHING
Phishing probability: 1.0000 -> PHISHING
Phishing probability: 1.0000 -> PHISHING
Phishing probability: 1.0000 -> PHISHING
Phishing probability: 1.0000 -> PHISHING
Phishing probability: 1.0000 -> PHISHING
Phishing probability: 1.0000 -> PHISHING
Phishing probability: 1.0000 -> PHISHING
Phishing probability: 1.0000 -> PHISHING


In [11]:
predict_url("https://accounts.google.com")
predict_url("https://paypal.com.security-update.ru")
predict_url("https://amazon.co.uk.login.verify-user.com")
predict_url("https://secure-login.microsoft.com.fake-domain.xyz")
predict_url("https://facebook.com-login-security-alert.com")
predict_url("https://update-appleid.apple.com.secure-check.xyz")
predict_url("https://login.live.com")
predict_url("https://github.com-login-user.com")

Phishing probability: 0.1505 -> LEGITIMATE
Phishing probability: 0.0245 -> LEGITIMATE
Phishing probability: 0.1268 -> LEGITIMATE
Phishing probability: 0.2759 -> LEGITIMATE
Phishing probability: 0.9193 -> PHISHING
Phishing probability: 1.0000 -> PHISHING
Phishing probability: 0.0563 -> LEGITIMATE
Phishing probability: 0.9947 -> PHISHING


In [13]:
predict_url("http://secure-login-paypal.com/account/update/verify/user/session/secure/index.php?id=12345")
predict_url("https://www.google.com/search?q=machine+learning")
predict_url("http://amazon.verify.account.security.update.login.com/index.php?user=abc123&session=xyz456")

Phishing probability: 1.0000 -> PHISHING
Phishing probability: 0.0047 -> LEGITIMATE
Phishing probability: 1.0000 -> PHISHING


In [14]:
predict_url("google.com")
predict_url("paypal.com")
predict_url("login-secure")
predict_url("http://localhost/login")
predict_url("https://test")
predict_url("http://example.com@phishing.com")

Phishing probability: 0.0005 -> LEGITIMATE
Phishing probability: 0.0005 -> LEGITIMATE
Phishing probability: 0.0149 -> LEGITIMATE
Phishing probability: 1.0000 -> PHISHING
Phishing probability: 0.0150 -> LEGITIMATE
Phishing probability: 1.0000 -> PHISHING
